# §0.2 確率・統計の復習 - 統計的多様体への準備

## 1. 概要

- **この節で学ぶこと**: パラメトリック分布族、指数型分布族、最尤推定、ベイズ推定の幾何学的準備
- **前提知識**: 確率分布、期待値、最尤推定の基礎
- **情報幾何との関連**: パラメータ空間が「統計的多様体」となり、指数型分布族は「双対平坦」という特別な構造を持つ

## 2. 直感的理解

### パラメータ空間を「地図」として考える

正規分布 $N(\mu, \sigma^2)$ を考えると：
- $(\mu, \sigma)$ の組が一つの「場所」
- 各場所に一つの確率分布が住んでいる
- この地図全体が「統計的多様体」

### 指数型分布族とは「特別に整った地図」

- 多くの重要な分布（正規、ベルヌーイ、ポアソン等）が指数型
- 2つの自然な座標系（θとη）がある
- ベイズ更新や最尤推定が「まっすぐな線」になる

## 3. 数学的定義

### 3.1 パラメトリック分布族

**定義**: パラメータ $\theta \in \Theta$ で添字付けられた確率分布の集合
$$\mathcal{S} = \{ p(x | \theta) : \theta \in \Theta \}$$

### 3.2 指数型分布族

**定義**: 以下の形で書ける分布族
$$p(x | \theta) = \exp\left[ \theta^\top T(x) - \psi(\theta) + h(x) \right]$$

- $\theta$: **自然パラメータ** (natural parameter)
- $T(x)$: **十分統計量** (sufficient statistic)
- $\psi(\theta)$: **対数分配関数** (log-partition function)
- $h(x)$: 基底測度

### 3.3 重要な関係式

**期待値パラメータ**: $\eta = \mathbb{E}[T(x)] = \nabla \psi(\theta)$

**Fisher情報行列**: $I(\theta) = \nabla^2 \psi(\theta) = \text{Cov}[T(x)]$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)

## 4. 可視化

### 4.1 パラメータ空間と分布の対応

In [ ]:
def visualize_parameter_space():
    """パラメータ空間の各点が一つの分布に対応することを可視化"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左図：パラメータ空間
    ax1 = axes[0]
    
    # 選んだ点を強調
    points = [
        (0, 1, 'red', 'A: N(0, 1)'),
        (2, 0.5, 'blue', 'B: N(2, 0.25)'),
        (-1, 1.5, 'green', 'C: N(-1, 2.25)'),
    ]
    
    # 背景のグリッド
    for mu in np.linspace(-2, 3, 10):
        for sigma in np.linspace(0.3, 2, 10):
            ax1.plot(mu, sigma, 'k.', markersize=2, alpha=0.3)
    
    for mu, sigma, color, label in points:
        ax1.plot(mu, sigma, 'o', color=color, markersize=12, label=label)
    
    ax1.set_xlabel('μ', fontsize=12)
    ax1.set_ylabel('σ', fontsize=12)
    ax1.set_title('Parameter space (Statistical manifold)\nEach point = one distribution', fontsize=11)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-2.5, 3.5)
    ax1.set_ylim(0, 2.5)
    
    # 右図：対応する分布
    ax2 = axes[1]
    x = np.linspace(-5, 6, 200)
    
    for mu, sigma, color, label in points:
        y = stats.norm.pdf(x, mu, sigma)
        ax2.plot(x, y, color=color, linewidth=2, label=label)
        ax2.fill_between(x, y, alpha=0.2, color=color)
    
    ax2.set_xlabel('x', fontsize=12)
    ax2.set_ylabel('p(x)', fontsize=12)
    ax2.set_title('Corresponding probability distributions', fontsize=11)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_parameter_space()

### 4.2 自然パラメータと期待値パラメータ（双対座標）

In [ ]:
def visualize_dual_coordinates():
    """ベルヌーイ分布での自然パラメータθと期待値パラメータηの関係"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bernoulli: θ = log(p/(1-p)), η = p = 1/(1+e^{-θ})
    theta = np.linspace(-4, 4, 100)
    eta = 1 / (1 + np.exp(-theta))  # シグモイド関数
    
    # 左図：θとηの関係
    ax1 = axes[0]
    ax1.plot(theta, eta, 'b-', linewidth=2)
    ax1.axhline(0.5, color='r', linestyle='--', alpha=0.5)
    ax1.axvline(0, color='r', linestyle='--', alpha=0.5)
    ax1.set_xlabel('θ (natural parameter)', fontsize=12)
    ax1.set_ylabel('η (mean parameter = p)', fontsize=12)
    ax1.set_title('Bernoulli: η = σ(θ) = 1/(1+e⁻ᶿ)', fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # 特徴的な点をマーク
    special_points = [(-2, 1/(1+np.exp(2))), (0, 0.5), (2, 1/(1+np.exp(-2)))]
    for t, e in special_points:
        ax1.plot(t, e, 'ro', markersize=8)
        ax1.annotate(f'({t}, {e:.2f})', (t, e), textcoords="offset points", 
                     xytext=(10, 10), fontsize=9)
    
    # 右図：対数分配関数
    ax2 = axes[1]
    psi = np.log(1 + np.exp(theta))  # ψ(θ) = log(1 + e^θ)
    dpsi = 1 / (1 + np.exp(-theta))  # ψ'(θ) = η
    
    ax2.plot(theta, psi, 'g-', linewidth=2, label='ψ(θ) = log(1+eᶿ)')
    ax2.plot(theta, dpsi, 'b--', linewidth=2, label="ψ'(θ) = η")
    ax2.set_xlabel('θ', fontsize=12)
    ax2.set_ylabel('Value', fontsize=12)
    ax2.set_title('Log-partition function and derivative', fontsize=11)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("""
【重要な関係式】
η = ∇ψ(θ)         ← 期待値パラメータは対数分配関数の勾配
I(θ) = ∇²ψ(θ)     ← Fisher情報 = 対数分配関数のヘッセ行列

これが「指数型分布族は双対平坦」の数学的根拠
""")

visualize_dual_coordinates()

## 5. 具体例

### 例1：正規分布の指数型表現

In [ ]:
print("""
【正規分布 N(μ, σ²) の指数型表現】

p(x|μ,σ) = (1/√(2πσ²)) exp(-(x-μ)²/(2σ²))
         = exp(μx/σ² - x²/(2σ²) - μ²/(2σ²) - log(σ√(2π)))

自然パラメータ: θ = (μ/σ², -1/(2σ²))
十分統計量:    T(x) = (x, x²)
対数分配関数:  ψ(θ) = -θ₁²/(4θ₂) - (1/2)log(-2θ₂) + (1/2)log(2π)

期待値パラメータ: η = (E[x], E[x²]) = (μ, μ² + σ²)
""")

# 数値確認
mu, sigma = 2, 1.5
theta1 = mu / sigma**2
theta2 = -1 / (2 * sigma**2)
print(f"μ={mu}, σ={sigma} の場合:")
print(f"  θ₁ = μ/σ² = {theta1:.4f}")
print(f"  θ₂ = -1/(2σ²) = {theta2:.4f}")
print(f"  η₁ = μ = {mu}")
print(f"  η₂ = μ² + σ² = {mu**2 + sigma**2}")

### 例2：最尤推定の幾何学的意味

In [ ]:
def demonstrate_mle_geometry():
    """最尤推定が期待値パラメータ座標での射影になることを示す"""
    np.random.seed(42)
    
    # 真のパラメータ
    true_mu, true_sigma = 2.0, 1.0
    
    # データ生成
    n = 50
    data = np.random.normal(true_mu, true_sigma, n)
    
    # MLE
    mle_mu = np.mean(data)
    mle_sigma = np.std(data, ddof=0)
    
    # 十分統計量
    T1 = np.mean(data)      # x̄
    T2 = np.mean(data**2)   # x̄²
    
    print(f"真のパラメータ: μ={true_mu}, σ={true_sigma}")
    print(f"MLE: μ̂={mle_mu:.4f}, σ̂={mle_sigma:.4f}")
    print(f"\n十分統計量（経験値）:")
    print(f"  T̄₁ = x̄ = {T1:.4f}")
    print(f"  T̄₂ = x̄² = {T2:.4f}")
    print(f"\n期待値パラメータ（MLE）:")
    print(f"  η̂₁ = μ̂ = {mle_mu:.4f}")
    print(f"  η̂₂ = μ̂² + σ̂² = {mle_mu**2 + mle_sigma**2:.4f}")
    print(f"\n指数型分布族では: η̂ = T̄ （期待値パラメータ = 十分統計量の平均）")

demonstrate_mle_geometry()

### 例3：ベイズ更新の可視化

In [ ]:
def visualize_bayesian_update():
    """正規分布の平均のベイズ更新（共役事前分布）"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 設定
    sigma_known = 1.0  # 既知の標準偏差
    mu_prior, tau_prior = 0, 2  # 事前分布
    observations = [1.5, 2.0, 1.8, 2.2]  # 観測値
    
    mu_range = np.linspace(-3, 5, 200)
    
    # 左図：分布の更新
    ax1 = axes[0]
    prior = stats.norm.pdf(mu_range, mu_prior, tau_prior)
    ax1.plot(mu_range, prior, 'k--', linewidth=2, label='Prior')
    
    mu_post, precision_post = mu_prior, 1/tau_prior**2
    trajectory = [(mu_post, 1/np.sqrt(precision_post))]
    
    colors = plt.cm.Blues(np.linspace(0.4, 1, len(observations)))
    for i, x in enumerate(observations):
        precision_likelihood = 1 / sigma_known**2
        precision_post_new = precision_post + precision_likelihood
        mu_post_new = (precision_post * mu_post + precision_likelihood * x) / precision_post_new
        
        precision_post, mu_post = precision_post_new, mu_post_new
        tau_post = 1 / np.sqrt(precision_post)
        trajectory.append((mu_post, tau_post))
        
        posterior = stats.norm.pdf(mu_range, mu_post, tau_post)
        ax1.plot(mu_range, posterior, color=colors[i], linewidth=1.5,
                 label=f'After obs {i+1}')
    
    ax1.set_xlabel('μ')
    ax1.set_ylabel('p(μ)')
    ax1.set_title('Sequential Bayesian update')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # 右図：パラメータ空間での軌跡
    ax2 = axes[1]
    trajectory = np.array(trajectory)
    ax2.plot(trajectory[:, 0], trajectory[:, 1], 'b.-', markersize=10, linewidth=2)
    ax2.plot(trajectory[0, 0], trajectory[0, 1], 'ks', markersize=12, label='Prior')
    ax2.plot(trajectory[-1, 0], trajectory[-1, 1], 'r^', markersize=12, label='Final posterior')
    
    ax2.set_xlabel('μ (mean)')
    ax2.set_ylabel('τ (std dev)')
    ax2.set_title('Trajectory in parameter space')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_bayesian_update()

## 6. 他の概念との関係

### 前の節との繋がり
- **線形代数 (§0.1)**: 正定値行列としてのFisher情報行列

### 次の節への接続
- **KLダイバージェンス (§0.3)**: 指数型分布族でKLが特に簡潔な形になる

### 情報幾何への応用

| 確率・統計の概念 | 情報幾何での対応 |
|----------------|----------------|
| パラメトリック分布族 | 統計的多様体 |
| 指数型分布族 | **双対平坦多様体** |
| 自然パラメータ θ | e-座標（指数座標）|
| 期待値パラメータ η | m-座標（混合座標）|
| 対数分配関数 ψ(θ) | ポテンシャル関数 |
| 最尤推定 | **m-射影** |
| ベイズ更新 | 多様体上の曲線 |

## 7. 演習問題

### Q1. 指数型分布族の表現

ポアソン分布 $p(x|\lambda) = \frac{\lambda^x e^{-\lambda}}{x!}$ を指数型分布族の標準形に書き直せ。

<details>
<summary>解答を見る</summary>

$$p(x|\lambda) = \exp(x \log \lambda - \lambda - \log x!)$$

- $\theta = \log \lambda$ （自然パラメータ）
- $T(x) = x$ （十分統計量）
- $\psi(\theta) = e^\theta = \lambda$ （対数分配関数）
- $h(x) = -\log x!$

期待値パラメータ: $\eta = \nabla\psi(\theta) = e^\theta = \lambda$

</details>

### Q2. 十分統計量

正規分布 $N(\mu, \sigma^2)$（両パラメータ未知）の十分統計量は何か？

<details>
<summary>解答を見る</summary>

$$T(x_1, \ldots, x_n) = \left( \sum_i x_i, \sum_i x_i^2 \right)$$

または等価的に $(\bar{x}, s^2)$ も十分統計量となる。

</details>

### Q3. Fisher情報と対数分配関数

ベルヌーイ分布で $\psi(\theta) = \log(1 + e^\theta)$ のとき、$\psi''(\theta)$ を計算し、これがFisher情報と一致することを確認せよ。

<details>
<summary>解答を見る</summary>

$$\psi'(\theta) = \frac{e^\theta}{1+e^\theta} = \frac{1}{1+e^{-\theta}} = p$$

$$\psi''(\theta) = \frac{e^\theta}{(1+e^\theta)^2} = p(1-p)$$

これは $\text{Var}[X] = p(1-p)$ と一致。Fisher情報 $I(\theta) = 1/(p(1-p))$ も $\psi''(\theta)$ から計算可能。

</details>

In [ ]:
# Q3の検証
theta = 1.0  # 例として
p = 1 / (1 + np.exp(-theta))
psi_double_prime = p * (1 - p)
variance = p * (1 - p)

print(f"θ = {theta}")
print(f"p = σ(θ) = {p:.4f}")
print(f"ψ''(θ) = {psi_double_prime:.4f}")
print(f"Var[X] = p(1-p) = {variance:.4f}")
print(f"一致: {np.isclose(psi_double_prime, variance)}")

## 8. 参考：使用したプロンプト

```
指数型分布族の定義と重要性を説明してください。
特に自然パラメータθと期待値パラメータηの関係を、
ベルヌーイ分布と正規分布を例に具体的に示してください。
```

```
最尤推定が「m-射影」になるとはどういう意味ですか？
指数型分布族での十分統計量との関係を含めて説明してください。
```

```
対数分配関数ψ(θ)のヘッセ行列がFisher情報行列になることを、
正規分布を例に手計算で確認してください。
```

---
**次のノートブック**: `03_kl_divergence_fisher.ipynb` - KLダイバージェンスとFisher情報